In [1]:
import pandas as pd 
import numpy as np
import sys

In [2]:
file_path = r'C:\dev\projects\NLP_test\data\URL_list.csv'
data = pd.read_csv(file_path)

In [3]:
import re
import json
from urllib.parse import urljoin, urlparse
from typing import Optional

import requests
from bs4 import BeautifulSoup, Tag

TIMEOUT = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

_SKIP_TAGS = {"nav", "footer", "header", "aside"}
_SKIP_ATTRS = re.compile(
    r"\b(nav|footer|header|menu|breadcrumb|sidebar|filter|facet|pagination|"
    r"cart|checkout|newsletter|popup|modal|cookie|banner|subscribe|"
    r"social|share|wishlist-modal|refinement)\b",
    re.IGNORECASE,
)

def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def _is_price(text: str) -> bool:
    return bool(re.match(r"^[\$€£¥₹]?\s?\d[\d,.\s]*$", text.strip()))

def _is_noise(text: str) -> bool:
    low = text.lower().strip()
    noise = {
        "quick view", "add to cart", "add to bag", "buy now", "enquire now",
        "added to wishlist", "view cart", "load more", "sort by", "refine by",
        "apply", "clear all", "close", "back", "menu", "help", "home",
        "professional", "submit", "check out", "enquire",
    }
    if low in noise:
        return True
    if len(text.strip()) <= 1:
        return True
    if re.match(r"^\d+$", text.strip()):
        return True
    return False


def _in_skip_zone(tag: Tag) -> bool:
    for parent in tag.parents:
        if not isinstance(parent, Tag):
            continue
        if parent.name in _SKIP_TAGS:
            return True
        cls = " ".join(parent.get("class", []))
        id_ = parent.get("id", "")
        role = parent.get("role", "")
        check = f"{cls} {id_} {role}"
        if _SKIP_ATTRS.search(check):
            return True
    return False

def _fetch(url: str) -> tuple[Optional[BeautifulSoup], Optional[str]]:
    try:
        resp = requests.get(
            url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True
        )
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml"), resp.url
    except Exception:
        return None, None


def _was_redirected_away(original_url: str, final_url: Optional[str]) -> bool:
    if final_url is None:
        return True
    orig = urlparse(original_url)
    final = urlparse(final_url)
    if orig.netloc.lower() != final.netloc.lower():
        return True
    if orig.path.rstrip("/") and final.path.rstrip("/") == "":
        return True
    return False

def _from_jsonld(soup: BeautifulSoup) -> list[str]:
    products: list[str] = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        _walk_ld(data, products)
    return products

def _walk_ld(node, out: list[str]):
    if isinstance(node, list):
        for item in node:
            _walk_ld(item, out)
        return
    if not isinstance(node, dict):
        return
    types = node.get("@type", "")
    types = types if isinstance(types, list) else [types]
    if any(t in ("Product", "IndividualProduct") for t in types):
        name = _clean(node.get("name", ""))
        if name:
            out.append(name)
        return
    if "ItemList" in types:
        for elem in node.get("itemListElement", []):
            _walk_ld(elem, out)
    if "@graph" in node:
        _walk_ld(node["@graph"], out)
    for key, val in node.items():
        if key.startswith("@"):
            continue
        if isinstance(val, (dict, list)):
            _walk_ld(val, out)

_PRODUCT_URL_PATTERNS = [
    re.compile(r"/[A-Z0-9][\w-]*\.html$", re.IGNORECASE),
    re.compile(r"/products/[\w-]+$", re.IGNORECASE),
    re.compile(r"/product/[\w-]+/?$", re.IGNORECASE),
    re.compile(r"/(?:p|item|dp)/[\w-]+/?$", re.IGNORECASE),
]

def _looks_like_product_url(href: str, base_host: str) -> bool:
    parsed = urlparse(href)
    if parsed.netloc and parsed.netloc != base_host:
        return False
    path = parsed.path
    for pat in _PRODUCT_URL_PATTERNS:
        if pat.search(path):
            return True
    return False


def _name_from_img_link(a_tag: Tag) -> Optional[str]:
    img = a_tag.find("img")
    if not img:
        return None
    for attr in ("title", "alt"):
        val = _clean(img.get(attr, "") or "")
        if val and not _is_noise(val) and not _is_price(val) and len(val) > 2:
            return val
    return None


def _name_from_text_link(a_tag: Tag) -> Optional[str]:
    lines = [_clean(l) for l in a_tag.get_text("\n").split("\n") if _clean(l)]
    for line in lines:
        if not _is_noise(line) and not _is_price(line) and len(line) > 2:
            return line
    return None

def _from_product_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    base_host = urlparse(base_url).netloc
    found: dict[str, str] = {}

    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        norm = urlparse(href)._replace(query="", fragment="").geturl()

        if not _looks_like_product_url(href, base_host):
            continue
        if _in_skip_zone(a):
            continue
        if norm in found:
            continue

        name = _name_from_img_link(a) or _name_from_text_link(a)
        if name:
            found[norm] = name

    return list(found.values())

_CARD_PATTERN = re.compile(
    r"\b(product[-_]?card|product[-_]?tile|product[-_]?item|"
    r"grid[-_]?item|collection[-_]?item|plp[-_]item|"
    r"ProductCard|productCard)\b",
    re.IGNORECASE,
)


def _name_from_card(card: Tag) -> Optional[str]:
    for level in ("h2", "h3", "h4", "h1", "h5"):
        h = card.find(level)
        if h:
            txt = _clean(h.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt
    for el in card.find_all(True):
        cls = " ".join(el.get("class", []))
        if re.search(r"(product|item|card)[_-]?(name|title)", cls, re.I):
            txt = _clean(el.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt
    a = card.find("a")
    if a:
        return _name_from_text_link(a)
    return None

def _from_html_cards(soup: BeautifulSoup) -> list[str]:
    products: list[str] = []
    seen: set[str] = set()

    candidates = [
        tag for tag in soup.find_all(True)
        if _CARD_PATTERN.search(" ".join(tag.get("class", [])))
    ]
    filtered: list[Tag] = []
    for c in candidates:
        if not any(c in f.descendants for f in filtered):
            filtered = [f for f in filtered if f not in c.descendants]
            filtered.append(c)

    for card in filtered:
        if _in_skip_zone(card):
            continue
        name = _name_from_card(card)
        if name and name.lower() not in seen:
            seen.add(name.lower())
            products.append(name)

    return products

def _from_og_product(soup: BeautifulSoup) -> Optional[str]:
    og_type = soup.find("meta", property="og:type")
    og_title = soup.find("meta", property="og:title")
    if og_type and og_title:
        otype = (og_type.get("content", "") or "").lower()
        title = _clean(og_title.get("content", "") or "")
        if "product" in otype and title:
            return title
    return None


def _from_single_product_page(soup: BeautifulSoup) -> Optional[str]:
    buy = soup.find(
        True,
        attrs={
            "class": re.compile(
                r"add[-_]?to[-_]?cart|buy[-_]?now|add[-_]?to[-_]?bag|"
                r"addtocart|product-form|shopify-payment",
                re.I,
            )
        },
    )
    if not buy:
        buy = soup.find(
            "button",
            string=re.compile(r"add to cart|buy now|add to bag|enquire", re.I),
        )
    price = soup.find(True, attrs={"class": re.compile(r"\bprice\b", re.I)})

    if buy or price:
        h1 = soup.find("h1")
        if h1:
            txt = _clean(h1.get_text())
            if txt and not _is_price(txt) and not _is_noise(txt):
                return txt

    og = _from_og_product(soup)
    if og:
        return og

    return None


def _from_title(soup: BeautifulSoup) -> Optional[str]:
    title_tag = soup.find("title")
    if title_tag:
        raw = _clean(title_tag.get_text())
        txt = re.split(r"\s*[|–—]\s*", raw)[0].strip()
        txt = re.sub(r"\s*[-–]\s*(Shop|Store|Buy|Online).*$", "", txt, flags=re.I).strip()
        if txt and not _is_noise(txt):
            return txt
    return None

def scrape_products(url: str) -> list[str]:
    soup, final_url = _fetch(url)

    if soup is None:
        return []

    # if _was_redirected_away(url, final_url):
    #     return []

    products = _from_jsonld(soup)
    if products:
        return products

    link_products = _from_product_links(soup, url)

    card_products = _from_html_cards(soup)

    catalog = link_products if len(link_products) >= len(card_products) else card_products

    if len(catalog) >= 2:
        return catalog

    single = _from_single_product_page(soup)
    if single:
        return [single]

    if catalog:
        return catalog
        
    title = _from_title(soup)
    if title:
        return [title]


In [9]:
urls = data.iloc[150:201, 0]
target_products = []

In [10]:
for url in urls:
    products = scrape_products(url)
    target_products.append(products)

In [11]:
clean_products = [
    ', '.join(product) if isinstance(product, (list, tuple)) else str(product) 
    for product in target_products
]

In [12]:
data_50 = data.iloc[150:201]

In [13]:
train_dataset = data_50.copy()
train_dataset['target'] = clean_products

In [14]:
train_dataset.head(10)

,max(page),target
150,https://www.scotmeachamwoodhome.com/products/h...,"Charlotte Crewelwork in Lake District Blue, Ma..."
151,https://www.espasso.com/products/tearsheet/1815,None
152,https://sika-design.com/products/belladonna-na...,Products
153,https://www.dimshome.com/products/eave-desk,None
154,https://thebellacottage.com/products/barrel-bar,Songbird Toile Settee Sofas & Settees Farmhous...
155,https://www.skandium.com/products/ch24-soft,CH24 Wishbone Chair - Beech - Soft colours - S...
156,https://www.woodwaves.com/products/gray-floati...,Floating TV Stand Wall Mount Entertainment Cen...
157,https://sjotime.com/products/notch-desk,
158,https://www.puji.com/www.puji.com/products/urb...,
159,https://littlepartners.com/products/bentwood,"Skip to content, The Learning Tower® Toddler T..."


In [15]:
train_dataset.to_csv(r'C:\dev\projects\NLP_test\data\train_dataset_50_tmp.csv', index=False, encoding='utf-8')